# 🚗 Esteira de Aprendizado de Maquina — Car Evaluation

**Dataset:** [Car Evaluation — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/19/car+evaluation)  
**Problema:** Classificacao multiclasse — avaliar um carro como `unacceptable`, `acceptable`, `good` ou `vgood`  
**Modelo:** Random Forest Classifier

---

## Etapas da Esteira

| # | Etapa |
|---|---|
| 1 | Carregamento dos dados |
| 2 | Estatisticas descritivas gerais |
| 3 | Transformacoes em colunas (features) |
| 4 | Transformacoes em linhas (duplicatas e outliers) |
| 5 | Divisao treino / validacao / teste (70 / 15 / 15) |
| 6 | Treinamento do modelo |
| 7 | Avaliacao — Matriz de Confusao e Acuracia |
| 8 | Predicao com o modelo implantado |


In [ ]:
# ── 0. IMPORTS ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="Set2")
print("Bibliotecas carregadas com sucesso.")


## 1. Carregamento dos Dados

O **Car Evaluation Dataset** foi criado por Marko Bohanec e e disponibilizado no UCI Machine Learning Repository.

- **1.728 instancias**
- **6 atributos de entrada** — todos categoricos
- **1 variavel-alvo** multiclasse: `unacceptable`, `acceptable`, `good`, `vgood`
- **Sem valores ausentes**

O dataset avalia carros com base em caracteristicas como preco de compra, custo de manutencao, numero de portas, capacidade de passageiros, tamanho do porta-malas e nivel de seguranca.


In [ ]:
URL = "https://raw.githubusercontent.com/YBIFoundation/Dataset/main/Car%20Evaluation.csv"

df_raw = pd.read_csv(URL)

# Renomeia colunas para nomes sem espacos
df_raw.columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]

print("Shape:", df_raw.shape)
print()
print("Primeiras linhas:")
display(df_raw.head(10))


## 2. Estatisticas Descritivas Gerais

In [ ]:
print("=" * 55)
print("INFORMACOES DO DATASET")
print("=" * 55)
df_raw.info()

print()
print("=" * 55)
print("VALORES AUSENTES POR COLUNA")
print("=" * 55)
print(df_raw.isnull().sum())

print()
print("=" * 55)
print("VALORES UNICOS POR COLUNA")
print("=" * 55)
for col in df_raw.columns:
    print(f"  {col:12s}: {sorted(df_raw[col].unique())}")

print()
print("=" * 55)
print("DISTRIBUICAO DA VARIAVEL-ALVO")
print("=" * 55)
vc = df_raw["class"].value_counts()
print(vc)
print()
pct = df_raw["class"].value_counts(normalize=True) * 100
for cls, p in pct.items():
    print(f"  {cls:15s}: {p:.1f}%")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

cols = ["buying", "maint", "doors", "persons", "lug_boot", "safety"]
titles = [
    "Preco de Compra", "Custo de Manutencao", "N de Portas",
    "Capacidade de Pessoas", "Tamanho do Porta-Malas", "Nivel de Seguranca"
]

palette = {"unacceptable": "#e41a1c", "acceptable": "#4daf4a",
           "good": "#377eb8", "vgood": "#ff7f00"}

for i, (col, title) in enumerate(zip(cols, titles)):
    ct = pd.crosstab(df_raw[col], df_raw["class"])
    ct.plot(kind="bar", ax=axes[i], color=[palette.get(c, "#999") for c in ct.columns],
            edgecolor="white", width=0.7)
    axes[i].set_title(title, fontweight="bold", fontsize=11)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Contagem")
    axes[i].tick_params(axis="x", rotation=30)
    axes[i].legend(title="Classe", fontsize=8)

plt.suptitle("Distribuicao das Classes por Feature", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
vc = df_raw["class"].value_counts()
colors = ["#e41a1c", "#4daf4a", "#377eb8", "#ff7f00"]
bars = ax.bar(vc.index, vc.values, color=colors, edgecolor="white", width=0.55)
ax.set_title("Distribuicao da Variavel-Alvo (class)", fontsize=13, fontweight="bold")
ax.set_ylabel("Contagem")
ax.set_xlabel("Classe")
for bar, v in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 10, str(v),
            ha="center", fontweight="bold", fontsize=11)
plt.tight_layout()
plt.show()
print("Nota: dataset desbalanceado — 'unacceptable' representa ~70% dos registros.")


## 3. Transformacoes nas Colunas (Features)

Como **todos os atributos sao categoricos e ordinais**, as transformacoes aplicadas foram:

| Transformacao | Coluna(s) | Motivo |
|---|---|---|
| Ordinal Encoding com ordem logica | `buying`, `maint`, `lug_boot`, `safety` | Preserva a ordem semantica das categorias (low < med < high < vhigh) |
| Ordinal Encoding com ordem logica | `doors`, `persons` | Preserva ordem numerica dentro da categoria |
| Label Encoding | `class` (target) | Converte o alvo em numerico para o modelo |
| Feature Engineering | `price_score` | Combina `buying` + `maint` em uma variavel de custo total |


In [ ]:
df = df_raw.copy()

# 3.1 Ordinal Encoding com ordens semanticas corretas
buying_order  = ["low", "med", "high", "vhigh"]
maint_order   = ["low", "med", "high", "vhigh"]
doors_order   = ["2", "3", "4", "5more"]
persons_order = ["2", "4", "more"]
boot_order    = ["small", "med", "big"]
safety_order  = ["low", "med", "high"]

ordinal_map = {
    "buying":   {v: i for i, v in enumerate(buying_order)},
    "maint":    {v: i for i, v in enumerate(maint_order)},
    "doors":    {v: i for i, v in enumerate(doors_order)},
    "persons":  {v: i for i, v in enumerate(persons_order)},
    "lug_boot": {v: i for i, v in enumerate(boot_order)},
    "safety":   {v: i for i, v in enumerate(safety_order)},
}

for col, mapping in ordinal_map.items():
    df[col] = df[col].map(mapping)

print("Mapeamentos ordinais aplicados:")
for col, mapping in ordinal_map.items():
    print(f"  {col:10s}: {mapping}")

# 3.2 Feature Engineering: custo total (buying + maint normalizado 0-1)
df["price_score"] = (df["buying"] + df["maint"]) / (3 + 3)  # max soma = 6
print()
print("Feature 'price_score' criada (0 = mais barato, 1 = mais caro)")
print(df["price_score"].describe().round(3))

# 3.3 Label Encoding do target
class_order = ["unacceptable", "acceptable", "good", "vgood"]
class_map   = {v: i for i, v in enumerate(class_order)}
df["class_enc"] = df["class"].map(class_map)

print()
print("Mapeamento do target:", class_map)
print()
print("Dataset apos transformacoes:")
display(df.head(8))


## 4. Transformacoes nas Linhas

| Transformacao | Criterio | Resultado |
|---|---|---|
| Remocao de duplicatas exatas | Linhas identicas em todas as colunas | Evita vies no treinamento |
| Remocao de combinacoes inconsistentes | `persons == 0` apos encoding (valor invalido) | Garante integridade dos dados |


In [ ]:
antes = len(df)

# 4.1 Remocao de duplicatas
df.drop_duplicates(inplace=True)
print(f"Linhas originais    : {antes}")
print(f"Duplicatas removidas: {antes - len(df)}")

# 4.2 Consistencia: persons encoding nao pode ser negativo
antes2 = len(df)
df = df[df["persons"] >= 0]
print(f"Linhas inconsistentes removidas: {antes2 - len(df)}")

print(f"Registros finais    : {len(df)}")
print()

# Distribuicao do target apos limpeza
print("Distribuicao do target apos limpeza:")
print(df["class"].value_counts())


## 5. Divisao em Treino / Validacao / Teste

Divisao **estratificada** para manter a proporcao das classes em cada subconjunto (importante pois o dataset e desbalanceado).

- **70%** Treino  
- **15%** Validacao  
- **15%** Teste  


In [ ]:
feature_cols = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "price_score"]

X = df[feature_cols]
y = df["class_enc"]

# Passo 1: 70% treino, 30% temporario
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Passo 2: 30% dividido em validacao (50%) e teste (50%) → 15% + 15%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

total = len(X)
print(f"{'Subconjunto':<12} | {'Amostras':>8} | {'Percentual':>10}")
print("-" * 38)
print(f"{'Treino':<12} | {len(X_train):>8} | {len(X_train)/total:>9.1%}")
print(f"{'Validacao':<12} | {len(X_val):>8} | {len(X_val)/total:>9.1%}")
print(f"{'Teste':<12} | {len(X_test):>8} | {len(X_test)/total:>9.1%}")

print()
class_names = ["unacceptable", "acceptable", "good", "vgood"]
inv_map = {0: "unacceptable", 1: "acceptable", 2: "good", 3: "vgood"}

print("Distribuicao de classes por subconjunto:")
for split_name, y_split in [("Treino", y_train), ("Validacao", y_val), ("Teste", y_test)]:
    vc = y_split.value_counts().sort_index()
    pcts = {inv_map[k]: f"{v/len(y_split)*100:.1f}%" for k, v in vc.items()}
    print(f"  {split_name:10s}: {pcts}")


## 6. Treinamento do Modelo — Random Forest Classifier

O **Random Forest** e especialmente adequado para este dataset porque:

- Lida naturalmente com features ordinais sem necessidade de normalizacao
- Robusto ao desbalanceamento de classes (com `class_weight='balanced'`)
- Fornece importancia de features interpretavel
- Multiclasse nativamente (sem precisar de estrategia One-vs-Rest)


In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,       # arvores completas
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Modelo treinado com sucesso!")
print(f"  Arvores: {model.n_estimators}")
print(f"  Classes: {model.classes_} -> {[inv_map[c] for c in model.classes_]}")

# Avaliacao no conjunto de validacao
y_val_pred = model.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
print()
print(f"Acuracia na Validacao: {val_acc*100:.2f}%")
print()
print("Relatorio de Classificacao — Validacao:")
print(classification_report(y_val, y_val_pred, target_names=class_names))


In [ ]:
# Grafico de importancia das features
feat_imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#fc8d62" if v > feat_imp.median() else "#8da0cb" for v in feat_imp]
feat_imp.plot(kind="barh", ax=ax, color=colors, edgecolor="white", width=0.6)
ax.set_title("Importancia das Features — Random Forest", fontsize=13, fontweight="bold")
ax.set_xlabel("Importancia Media (Gini)")
ax.axvline(feat_imp.median(), color="gray", linestyle="--", alpha=0.6, label="Mediana")
ax.legend()
plt.tight_layout()
plt.show()


## 7. Avaliacao Final — Matriz de Confusao e Acuracia (Conjunto de Teste)

O conjunto de **teste** e utilizado somente nesta etapa para uma avaliacao imparcial do modelo.


In [ ]:
y_test_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)

print("=" * 55)
print("AVALIACAO NO CONJUNTO DE TESTE")
print("=" * 55)
print(f"Acuracia: {test_acc*100:.2f}%")
print()
print("Relatorio de Classificacao:")
print(classification_report(y_test, y_test_pred, target_names=class_names))

# Matriz de Confusao
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=25)
ax.set_title(
    f"Matriz de Confusao — Conjunto de Teste  |  Acuracia: {test_acc*100:.1f}%",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Analise por classe
print("Analise por classe:")
print(f"  {'Classe':<15} {'Corretos':>8} {'Total':>8} {'Acuracia':>10}")
print("  " + "-" * 45)
for i, cls in enumerate(class_names):
    mask = (y_test == i)
    n_total = mask.sum()
    n_correct = ((y_test == i) & (y_test_pred == i)).sum()
    acc_cls = n_correct / n_total if n_total > 0 else 0
    print(f"  {cls:<15} {n_correct:>8} {n_total:>8} {acc_cls:>9.1%}")


## 8. Predicao com o Modelo Implantado

Cenario: dado um carro com caracteristicas especificas, o modelo retorna a classificacao e as probabilidades para cada classe.


In [ ]:
# Caracteristicas do carro a ser avaliado (valores originais)
carro = {
    "buying":   "med",    # preco de compra: low | med | high | vhigh
    "maint":    "low",    # manutencao:      low | med | high | vhigh
    "doors":    "4",      # portas:          2 | 3 | 4 | 5more
    "persons":  "4",      # pessoas:         2 | 4 | more
    "lug_boot": "big",    # porta-malas:     small | med | big
    "safety":   "high",   # seguranca:       low | med | high
}

print("Carro a ser avaliado:")
for k, v in carro.items():
    print(f"  {k:12s}: {v}")

# Aplicar os mesmos mapeamentos ordinais
paciente = pd.DataFrame([carro])
for col, mapping in ordinal_map.items():
    paciente[col] = paciente[col].map(mapping)

# Feature Engineering
paciente["price_score"] = (paciente["buying"] + paciente["maint"]) / 6.0

X_novo = paciente[feature_cols]

pred_enc   = model.predict(X_novo)[0]
pred_label = inv_map[pred_enc]
pred_proba = model.predict_proba(X_novo)[0]

print()
print("=" * 50)
print("       RESULTADO DA PREDICAO")
print("=" * 50)
print(f"  Classe predita: {pred_label.upper()}")
print()
print("  Probabilidades por classe:")
for cls, prob in zip(class_names, pred_proba):
    bar = "#" * int(prob * 40)
    print(f"  {cls:<15}: {prob:>5.1%}  {bar}")
print("=" * 50)


In [ ]:
# Testando 3 carros com perfis diferentes
casos = [
    {"buying": "vhigh", "maint": "vhigh", "doors": "2", "persons": "2",  "lug_boot": "small", "safety": "low",  "desc": "Caro, pequeno, inseguro"},
    {"buying": "low",   "maint": "low",   "doors": "4", "persons": "4",  "lug_boot": "big",   "safety": "high", "desc": "Barato, espacoso, seguro"},
    {"buying": "med",   "maint": "med",   "doors": "4", "persons": "more","lug_boot": "med",  "safety": "high", "desc": "Custo moderado, familiar"},
]

print(f"{'Descricao':<30} | {'Predicao':<15} | {'Prob. Predita':>13}")
print("-" * 65)
for caso in casos:
    desc = caso.pop("desc")
    p = pd.DataFrame([caso])
    for col, mapping in ordinal_map.items():
        p[col] = p[col].map(mapping)
    p["price_score"] = (p["buying"] + p["maint"]) / 6.0
    pred = model.predict(p[feature_cols])[0]
    prob = model.predict_proba(p[feature_cols])[0][pred]
    print(f"{desc:<30} | {inv_map[pred]:<15} | {prob:>12.1%}")
    caso["desc"] = desc  # restaura


## Conclusao

| Etapa | Descricao | Status |
|---|---|---|
| 1 | Carregamento dos dados (UCI via GitHub mirror) | OK |
| 2 | Estatisticas descritivas e visualizacoes | OK |
| 3 | Transformacoes em colunas (Ordinal Encoding + Feature Engineering) | OK |
| 4 | Transformacoes em linhas (duplicatas + inconsistencias) | OK |
| 5 | Divisao treino/validacao/teste 70/15/15 estratificada | OK |
| 6 | Treinamento Random Forest Classifier (300 arvores) | OK |
| 7 | Matriz de Confusao e Acuracia no conjunto de teste | OK |
| 8 | Predicao de novos carros com probabilidades por classe | OK |

---

**Dataset:** Car Evaluation — UCI Machine Learning Repository (1.728 instancias, 6 features, 4 classes)  
**Modelo:** Random Forest Classifier  
**Stack:** Python 3 · scikit-learn · pandas · numpy · matplotlib · seaborn
